# 02 — Feature Engineering

Applies the cleaning and encoding decisions discovered during EDA ([01_eda.ipynb](01_eda.ipynb)), builds the binary 30-day readmission target, splits the data, and constructs the preprocessing pipeline used by every model downstream.

All of the cleaning logic lives in [`src/data_prep.py`](../src/data_prep.py) and [`src/preprocessing.py`](../src/preprocessing.py) — this notebook documents *why* each step is needed and persists the resulting artifacts to `data/processed/` and `models/` so that 03/04/05 can load them directly without re-running the cleaning pipeline.

In [1]:
import sys
sys.path.append('..')

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=UserWarning)
warnings.simplefilter("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", message="Found unknown categories in columns")

import joblib
import pandas as pd
import numpy as np

from src import data_prep, preprocessing

## Load raw data and drop identifier columns

In [2]:
df = data_prep.load_data()
df = data_prep.drop_identifier_columns(df)
df.shape

(101766, 48)

## Drop low-information medication columns

EDA showed that for most medication columns, >99% of values fall into a single category (`modal_share` in [01_eda.ipynb](01_eda.ipynb)). These carry essentially no signal, so they're dropped along with `weight` (97% missing).

In [3]:
df = df.drop(columns=data_prep.LOW_INFORMATION_COLUMNS)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 30 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   race                      101766 non-null  object
 1   gender                    101766 non-null  object
 2   age                       101766 non-null  object
 3   admission_type_id         101766 non-null  int64 
 4   discharge_disposition_id  101766 non-null  int64 
 5   admission_source_id       101766 non-null  int64 
 6   time_in_hospital          101766 non-null  int64 
 7   payer_code                101766 non-null  object
 8   medical_specialty         101766 non-null  object
 9   num_lab_procedures        101766 non-null  int64 
 10  num_procedures            101766 non-null  int64 
 11  num_medications           101766 non-null  int64 
 12  number_outpatient         101766 non-null  int64 
 13  number_emergency          101766 non-null  int64 
 14  numb

## Clean admission/discharge/source ID columns

`admission_type_id`, `discharge_disposition_id`, and `admission_source_id` are numeric codes that map to readable categories (per the dataset documentation). Several codes correspond to "Not Available" / "NULL" / "Not Mapped" / "Unknown" — these are recoded to `NaN` once mapped to labels.

In [4]:
for col, mapping in data_prep.ID_MAPPINGS.items():
    df[col] = df[col].map(mapping)

for col in data_prep.ID_MAPPINGS:
    df[col] = df[col].replace(data_prep.NULL_LABELS, np.nan)

df[list(data_prep.ID_MAPPINGS)].head()

,admission_type_id,discharge_disposition_id,admission_source_id
0,NaN,NaN,Physician Referral
1,Emergency,Discharged to home,Emergency Room
2,Emergency,Discharged to home,Emergency Room
3,Emergency,Discharged to home,Emergency Room
4,Emergency,Discharged to home,Emergency Room


## Drop "Expired" discharges

Patients discharged as "Expired" cannot be readmitted — keeping them in the data would bias the negative class (`readmitted == 'NO'`).

In [5]:
before = len(df)
df = df[~df['discharge_disposition_id'].isin(data_prep.EXPIRED_DISCHARGE_LABELS)].reset_index(drop=True)
print(f"Dropped {before - len(df)} expired-discharge rows -> {len(df)} rows remain")

Dropped 1652 expired-discharge rows -> 100114 rows remain


## Recode literal `'?'` placeholders to missing

In [6]:
df = df.replace('?', np.nan)
df.isnull().sum()

race                         2239
gender                          0
age                             0
admission_type_id           10237
discharge_disposition_id     4680
admission_source_id          6929
time_in_hospital                0
payer_code                  39591
medical_specialty           49129
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                         21
diag_2                        358
diag_3                       1421
number_diagnoses                0
max_glu_serum               94890
A1Cresult                   83238
metformin                       0
repaglinide                     0
glipizide                       0
pioglitazone                    0
rosiglitazone                   0
insulin                         0
change                          0
diabetesMed                     0
readmitted    

## Group ICD-9 diagnosis codes

`diag_1`, `diag_2`, `diag_3` have 700+ distinct ICD-9 codes — far too high cardinality to one-hot encode directly. `data_prep.map_icd9_to_category` groups them into broad clinical categories (Circulatory, Respiratory, Diabetes, Injury, ...) following the ranges at https://www.aapc.com/codes/icd9-codes-range/.

In [7]:
for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].apply(data_prep.map_icd9_to_category)

df['diag_1'].value_counts()

diag_1
Circulatory             29782
Respiratory             10058
Digestive                9115
Other                    8458
Symptoms/Ill-defined     7534
Injury                   6881
Genitourinary            5015
Musculoskeletal          4944
Neoplasms                3299
Endocrine/Metabolic      2668
Skin                     2597
Infectious               2585
Mental                   2259
Supplementary            1638
Nervous System           1192
Blood                    1095
Pregnancy                 687
Diabetes                  235
Congenital                 50
Unknown                    21
External Causes             1
Name: count, dtype: int64

## Group medical specialties

`medical_specialty` has ~70 raw values where a handful dominate. `data_prep.map_medical_specialty` groups them into broader clinical groups (Cardiology, Surgery, Endocrinology, ...) by keyword matching.

In [8]:
df['medical_specialty'] = df['medical_specialty'].apply(data_prep.map_medical_specialty)
df['medical_specialty'].value_counts()

medical_specialty
Unknown               49129
General Practice      24698
Emergency              7449
Cardiology             6043
Orthopedics            2625
Nephrology             1544
Surgery                1246
Radiology              1182
Psychiatry              962
Pulmonology             881
OB/GYN                  746
Urology                 684
Oncology                556
Gastroenterology        550
Other                   398
Rehabilitation          391
Pediatrics              366
Endocrinology           286
Neurology               213
Hematology               78
Ophthalmology            38
Infectious Disease       37
Anesthesiology           12
Name: count, dtype: int64

## Build the binary 30-day readmission target

We only care about readmissions within 30 days, so `'NO'` and `'>30'` are combined into class 0, and `'<30'` becomes class 1.

In [9]:
df = data_prep.add_binary_target(df)
df['readmitted_30'].value_counts(normalize=True)

readmitted_30
0    0.886559
1    0.113441
Name: proportion, dtype: float64

## Sanity check against `src.data_prep`

The cell-by-cell cleaning above should produce exactly the same dataframe as the convenience function `data_prep.load_and_clean()`, which downstream notebooks will use directly.

In [10]:
df_check = data_prep.load_and_clean()
pd.testing.assert_frame_equal(df.reset_index(drop=True), df_check.reset_index(drop=True))
print("OK — matches src.data_prep.load_and_clean()")

OK — matches src.data_prep.load_and_clean()


## Train/test split

In [11]:
X, y = preprocessing.split_features_target(df)
X_train, X_test, y_train, y_test = preprocessing.make_train_test_split(X, y)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print("Train target balance:")
print(y_train.value_counts(normalize=True))

X_train: (80091, 29), X_test: (20023, 29)
Train target balance:
readmitted_30
0    0.886554
1    0.113446
Name: proportion, dtype: float64


## Build the preprocessing pipeline

- Numeric columns: median impute + standard scale
- Categorical columns: constant impute (`'Unknown'`) + one-hot encode
- `age`: ordinal encode using its natural bucket order (`[0-10)`, `[10-20)`, ...)

This `ColumnTransformer` is the first step of every model pipeline in 03/04/05 — fitting happens inside each model's pipeline (on the training fold only) to avoid leakage.

In [12]:
preprocessor = preprocessing.build_preprocessor(X)
num_cols, cat_cols, ord_cols = preprocessing.get_column_groups(X)

print("num_cols:", num_cols)
print("cat_cols:", cat_cols)
print("ord_cols:", ord_cols)
preprocessor

num_cols: ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
cat_cols: ['race', 'gender', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'glipizide', 'pioglitazone', 'rosiglitazone', 'insulin', 'change', 'diabetesMed']
ord_cols: ['age']


ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['time_in_hospital', 'num_lab_procedures',
                                  'num_procedures', 'num_medications',
                                  'number_outpatient', 'number_emergency',
                                  'number_inpatient', 'number_diagnoses']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Unkno...
                                  'medical_specialty', 'diag_1', 'diag_2',
                                  'diag_3', 'max_glu_serum', 'A1Cresult',
                                  'metformin', 'repaglinide', 'glipizide',
                                  'pioglitazone', 'rosiglitazone', 'insulin',
                                  'change', 'diabetesMed']),
                                ('ord',
                                 Pipeline(steps=[('encoder',
                                                  OrdinalEncoder(categories=[['[0-10)',
                                                                              '[10-20)',
                                                                              '[20-30)',
                                                                              '[30-40)',
                                                                              '[40-50)',
                                                                              '[50-60)',
                                                                              '[60-70)',
                                                                              '[70-80)',
                                                                              '[80-90)',
                                                                              '[90-100)']]))]),
                                 ['age'])])

## Persist artifacts for downstream notebooks

Saves the cleaned dataframe, the train/test split, and the (unfitted) preprocessor + column groups so that 03 (baseline models), 04 (tuning), and 05 (evaluation) can load them directly and run independently.

In [13]:
from pathlib import Path

PROCESSED_DIR = '../data/processed'
MODELS_DIR = '../models'
Path(PROCESSED_DIR).mkdir(parents=True, exist_ok=True)
Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)

df.to_parquet(f'{PROCESSED_DIR}/cleaned_data.parquet', index=False)

X_train.to_parquet(f'{PROCESSED_DIR}/X_train.parquet', index=False)
X_test.to_parquet(f'{PROCESSED_DIR}/X_test.parquet', index=False)
y_train.to_frame().to_parquet(f'{PROCESSED_DIR}/y_train.parquet', index=False)
y_test.to_frame().to_parquet(f'{PROCESSED_DIR}/y_test.parquet', index=False)

joblib.dump(preprocessor, f'{MODELS_DIR}/preprocessor.joblib')
joblib.dump({'num_cols': num_cols, 'cat_cols': cat_cols, 'ord_cols': ord_cols},
            f'{MODELS_DIR}/feature_columns.joblib')

print("Saved cleaned data, train/test split, preprocessor, and feature column groups.")

Saved cleaned data, train/test split, preprocessor, and feature column groups.


**Next**: [03_baseline_models.ipynb](03_baseline_models.ipynb) loads these artifacts and trains baseline classifiers with default hyperparameters.